# R21 Resting-State Randomise Results

Inspect every N=27 **cluster-extent corrected** primary and secondary randomise result on MNI anatomy, summarize the corresponding dual-regression stage-2 beta for sham, RTPJ, VLPFC, and BOTH, and review the condition-level MRIQC boxplots used to evaluate differential motion. Each result is shown in its own interactive NiiVue viewer. TFCE results are intentionally excluded.

## 1. Verify notebook dependencies

Launch this notebook with `bash notebooks/run_randomise_notebook.sh`. IPyNiiVue uses both a Python kernel package and an AnyWidget browser extension, so installing it after JupyterLab has already started is not sufficient.

In [ ]:
from pathlib import Path
import importlib.util

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'code' / 'check_randomise_results.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the r21-rest repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
required_imports = ('ipyniivue', 'ipywidgets', 'matplotlib', 'nibabel', 'nilearn', 'numpy', 'pandas')
missing = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. Close JupyterLab and run: bash notebooks/run_randomise_notebook.sh')
print('Notebook dependencies and IPyNiiVue kernel package are available.')
print('Project root:', PROJECT_ROOT)

## 2. Load significant cluster-extent results

FSL corrected-p images contain `1-p`; the notebook defines the displayed region as voxels with `1-p > 0.95`. The two tracked summary tables cover 77 primary and 98 secondary network-by-contrast jobs. Their corrected p-values control spatial inference within each job, not multiplicity across all 175 jobs.

In [ ]:
import json

import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, image
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from ipyniivue import NiiVue

CORRP_THRESHOLD = 0.95
CONDITION_ORDER = ['sham', 'rtpj', 'vlpfc', 'both']
CONDITION_LABELS = ['Sham', 'RTPJ', 'VLPFC', 'BOTH']
NETWORK_LABELS = {
    'dmn': 'default mode network (DMN)',
    'ecn': 'executive control network (ECN)',
    'right-fpn': 'right frontoparietal network',
    'left-fpn': 'left frontoparietal network',
    'primary-visual': 'primary visual network',
    'occipital-pole': 'occipital-pole visual network',
    'lateral-visual': 'lateral visual network',
    'primary-visual-lateral-visual': 'combined primary/lateral visual component',
    'sensorimotor': 'sensorimotor network',
    'auditory': 'auditory network',
}
ANALYSIS_LABELS = {'0': 'data-derived ICA, automatic dimensionality', '20': 'data-derived ICA, 20 components', 'smith09': 'direct Smith09 RSN map'}
SUMMARY_DIR = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary'
SUMMARY_SPECS = [
    ('primary', SUMMARY_DIR / 'task-rest_randomise_peak_summary.tsv'),
    ('secondary', SUMMARY_DIR / 'task-rest_desc-SecondaryNetworks_randomise_peak_summary.tsv'),
]
missing_summaries = [str(path) for _, path in SUMMARY_SPECS if not path.is_file()]
if missing_summaries:
    raise FileNotFoundError('Missing randomise summaries: ' + ', '.join(missing_summaries))

result_tables = []
for family, path in SUMMARY_SPECS:
    table = pd.read_csv(path, sep='\t', dtype=str).fillna('')
    table['family'] = family
    table['summary_file'] = str(path.relative_to(PROJECT_ROOT))
    result_tables.append(table)
results = pd.concat(result_tables, ignore_index=True)

if 'roi_values_tsv' not in results.columns:
    raise RuntimeError('Rerun code/check_randomise_results.py on the Linux box and push the generated ROI-value TSVs.')
significant = results.loc[(results['inference'] == 'cluster-extent') & (results['peak_gt_threshold'] == 'true') & (results['status'] == 'ok')].copy()
contrast_terms = {
    'both-minus-sham': ('BOTH', 'sham'),
    'both-minus-rtpj': ('BOTH', 'RTPJ'),
    'both-minus-vlpfc': ('BOTH', 'VLPFC'),
    'rtpj-minus-vlpfc': ('RTPJ', 'VLPFC'),
    'rtpj-minus-sham': ('RTPJ', 'sham'),
    'vlpfc-minus-sham': ('VLPFC', 'sham'),
    'both-minus-mean-rtpj-vlpfc': ('BOTH', 'average(RTPJ, VLPFC)'),
}
def signed_effect(row):
    first, second = contrast_terms[row['condition_contrast']]
    if row['direction'] == 'negative':
        first, second = second, first
    return f'{first} > {second}'

significant['effect'] = significant.apply(signed_effect, axis=1)
significant['peak_fwe_p'] = 1.0 - significant['peak_corrp'].astype(float)
significant['family'] = pd.Categorical(significant['family'], ['primary', 'secondary'], ordered=True)
significant = significant.sort_values(['family', 'peak_fwe_p', 'analysis', 'network']).reset_index(drop=True)
if significant.empty:
    raise RuntimeError('The summary contains no complete cluster-extent maps with peak 1-p > 0.95.')
if (significant['roi_values_tsv'].str.strip() == '').any():
    raise RuntimeError('Portable ROI values are missing. Rerun code/check_randomise_results.py on Linux, then commit and push derivatives/fsl/randomise_summary.')
significant['analysis_description'] = significant['analysis'].map(ANALYSIS_LABELS)
significant['network_description'] = significant['network'].map(NETWORK_LABELS).fillna(significant['network'])

QC_DIR = PROJECT_ROOT / 'derivatives' / 'qc'
QC_CONTRASTS = pd.read_csv(QC_DIR / 'task-rest_mriqc_condition_contrasts.tsv', sep='\t', dtype=str).fillna('')
QC_BOUNDS = pd.read_csv(QC_DIR / 'task-rest_qc_sensitivity_abs_bounds.tsv', sep='\t')
QC_SELECTION = pd.read_csv(QC_DIR / 'task-rest_qc_sensitivity_exclusions.tsv', sep='\t', dtype=str).fillna('')

def paired_motion_ids(condition_contrast):
    key = condition_contrast.replace('-', '_')
    mask = QC_SELECTION['paired_motion_outlier_contrasts'].str.split(',').apply(lambda values: key in values)
    return QC_SELECTION.loc[mask, 'participant'].tolist()

significant['paired_motion_flags'] = significant['condition_contrast'].apply(lambda value: ', '.join(paired_motion_ids(value)) or 'none')
summary_table = significant[['family', 'analysis_description', 'network_description', 'component', 'effect', 'peak_fwe_p', 'paired_motion_flags']].copy()
summary_table.columns = ['Family', 'Analysis', 'Network', 'Map/component', 'Signed effect', 'Peak FWE-corrected p', 'Paired motion flags']
summary_table['Family'] = summary_table['Family'].astype(str).str.title()
summary_table.insert(0, 'Result', range(1, len(summary_table) + 1))
primary_count = int((significant['family'] == 'primary').sum())
secondary_count = int((significant['family'] == 'secondary').sum())
print(f'Cluster-extent results available: {len(significant)} ({primary_count} primary; {secondary_count} secondary)')
print('No result survives a simple Bonferroni correction across all 175 network-by-contrast jobs.')
display(summary_table.style.format({'Peak FWE-corrected p': '{:.4f}'}).hide(axis='index'))

## 3. Define visualization and ROI-summary helpers

In [ ]:
def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

def result_image(row):
    copied = str(row.get('copied_image', '')).strip()
    source = copied or str(row['corrp_file']).strip()
    path = project_path(source)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def result_label(row):
    return f"{row['effect']} in the {row['network_description']}"

def result_sidecar(row):
    path = project_path(row['copied_sidecar'])
    return json.loads(path.read_text()) if path.is_file() else {}

def extract_condition_values(row_index):
    row = significant.loc[row_index]
    corrp_img = nib.load(str(result_image(row)))
    corrp_data = np.asarray(corrp_img.dataobj)
    roi = np.isfinite(corrp_data) & (corrp_data > CORRP_THRESHOLD)
    if not roi.any():
        raise ValueError('Selected corrected-p map has no voxels above the threshold.')

    values_path = project_path(row['roi_values_tsv'])
    if not values_path.is_file():
        raise FileNotFoundError(f'Portable ROI-value table is not tracked locally: {values_path}')
    values = pd.read_csv(values_path, sep='\t', dtype={'participant': str, 'condition': str})
    required = {'participant', 'condition', 'stage2_beta'}
    if not required.issubset(values.columns):
        raise ValueError(f'{values_path} is missing columns: {sorted(required.difference(values.columns))}')
    values['stage2_beta'] = pd.to_numeric(values['stage2_beta'], errors='raise')
    values['condition'] = values['condition'].str.lower()
    counts = values.groupby('participant')['condition'].nunique()
    complete = counts[counts == 4].index
    values = values.loc[values['participant'].isin(complete)].copy()
    values['condition'] = pd.Categorical(values['condition'], CONDITION_ORDER, ordered=True)
    return values, corrp_img, roi

def plot_condition_bars(values, title):
    summary = values.groupby('condition', observed=False)['stage2_beta'].agg(['mean', 'sem', 'count']).reindex(CONDITION_ORDER)
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    colors = ['#8A8F98', '#2878B5', '#D95F59', '#3A9D6F']
    positions = np.arange(len(CONDITION_ORDER))
    ax.bar(positions, summary['mean'], yerr=summary['sem'], capsize=5, color=colors, edgecolor='#222222', linewidth=0.8, alpha=0.86)
    for position, condition in zip(positions, CONDITION_ORDER):
        points = values.loc[values['condition'] == condition, 'stage2_beta'].sort_values().to_numpy()
        jitter = np.linspace(-0.12, 0.12, len(points))
        ax.scatter(position + jitter, points, s=17, color='#202020', alpha=0.48, linewidth=0, zorder=3)
    ax.set_xticks(positions, CONDITION_LABELS)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.set_ylabel('Dual-regression stage-2 beta\n(mean ± SEM)')
    ax.set_title(title, fontsize=12, pad=12)
    ax.text(0.99, 0.02, f"n = {int(summary['count'].min())} participants", transform=ax.transAxes, ha='right', va='bottom', color='#555555', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    return summary

## 4. All primary and secondary significant results

Each warm-colored overlay contains only voxels with cluster-extent FWE-corrected `1-p > 0.95` (corrected `p < 0.05`). The overlay encodes corrected significance, not connectivity magnitude. Use the NiiVue controls to change slice position, orientation, opacity, and rendering mode.

In [ ]:
CACHE_DIR = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary' / '.notebook_cache'
CACHE_DIR.mkdir(exist_ok=True)
mni_img = datasets.load_mni152_template(resolution=2)
mni_path = Path(mni_img.get_filename()) if mni_img.get_filename() else CACHE_DIR / 'MNI152_T1_2mm.nii.gz'
if not mni_path.is_file():
    nib.save(mni_img, mni_path)

def thresholded_map_path(row, corrp_img, roi):
    source = result_image(row)
    destination = CACHE_DIR / source.name.replace('_stat-corrp_statmap.nii.gz', '_desc-thresholded_statmap.nii.gz')
    thresholded = image.new_img_like(corrp_img, np.where(roi, np.asarray(corrp_img.dataobj), 0.0), copy_header=True)
    nib.save(thresholded, destination)
    return destination

for result_number, (row_index, row) in enumerate(significant.iterrows(), start=1):
    values, corrp_img, roi = extract_condition_values(row_index)
    sidecar = result_sidecar(row)
    corrected_p = 1.0 - float(row['peak_corrp'])
    n_participants = values['participant'].nunique()
    motion_ids = paired_motion_ids(row['condition_contrast'])
    motion_text = ', '.join(f'`{participant}`' for participant in motion_ids) if motion_ids else 'none'
    map_label = f"map {int(row['component'])}" if row['analysis'] == 'smith09' else f"component {int(row['component'])}"
    display(Markdown(
        f"### Result {result_number}: {result_label(row)}\n\n"
        f"- **Analysis family:** {str(row['family']).title()}.\n"
        f"- **Network definition:** {row['analysis_description']}, {map_label}.\n"
        f"- **Tested contrast:** `{row['condition_contrast']}`, {row['design_contrast']} ({row['direction']}); this is **{row['effect']}**.\n"
        f"- **Inference:** one-sample randomise, {int(sidecar.get('NumberOfPermutations', 5000)):,} permutations, cluster-forming *t* = {float(sidecar.get('ClusterFormingTThreshold', 3.1)):.1f}.\n"
        f"- **Corrected result:** peak FWE-corrected *p* = {corrected_p:.4f}; {int(roi.sum())} suprathreshold voxels.\n"
        f"- **Differential-motion context:** participants with paired mean-FD and high-motion-percentage boxplot flags for this comparison: {motion_text}.\n"
        f"- **Bar plot:** mean stage-2 beta ± SEM across {n_participants} participants within the displayed cluster."
    ))
    overlay_path = thresholded_map_path(row, corrp_img, roi)
    viewer = NiiVue()
    viewer.load_volumes([
        {'path': str(mni_path), 'name': 'MNI152 T1 2 mm', 'colormap': 'gray', 'opacity': 1.0},
        {'path': str(overlay_path), 'name': row['effect'], 'colormap': 'red', 'opacity': 0.80, 'cal_min': CORRP_THRESHOLD, 'cal_max': 1.0},
    ])
    display(viewer)
    plot_condition_bars(values, result_label(row))

## 5. Differential data-quality boxplots

The N=27 analysis contains participants with one usable run in each condition. Starting from 31 participants, `sub-216` and `sub-232` have no task-rest BOLD, `sub-212` has only SHAM and VLPFC runs, and `sub-233` has four BOLD runs but zero-row placeholder events files with no condition values. No participant was excluded from this N=27 analysis for motion or tSNR.

These plots show the **absolute within-participant condition difference** for every planned comparison. Each gray point is one participant. Orange points exceed that panel's Tukey upper fence. In the two motion panels, red points exceed both the absolute mean-FD and absolute high-motion-percentage fences for the same comparison. The short red marks show the computed upper fences.

In [ ]:
from matplotlib.lines import Line2D

QC_CONTRAST_ORDER = [
    'both_minus_sham',
    'both_minus_rtpj',
    'both_minus_vlpfc',
    'rtpj_minus_vlpfc',
    'rtpj_minus_sham',
    'vlpfc_minus_sham',
    'both_minus_mean_rtpj_vlpfc',
]
QC_CONTRAST_LABELS = [
    'BOTH − SHAM',
    'BOTH − RTPJ',
    'BOTH − VLPFC',
    'RTPJ − VLPFC',
    'RTPJ − SHAM',
    'VLPFC − SHAM',
    'BOTH − mean(RTPJ, VLPFC)',
]
ANALYSIS_PARTICIPANTS = set(QC_SELECTION['participant'])
complete_qc = QC_CONTRASTS.loc[
    (QC_CONTRASTS['complete'].str.lower() == 'true')
    & QC_CONTRASTS['participant'].isin(ANALYSIS_PARTICIPANTS)
].copy()
for column in ('delta_tsnr', 'delta_fd_mean', 'delta_fd_perc'):
    complete_qc[column] = pd.to_numeric(complete_qc[column], errors='raise')

def absolute_fence(contrast, metric):
    row = QC_BOUNDS.loc[(QC_BOUNDS['contrast'] == contrast) & (QC_BOUNDS['metric'] == metric)]
    if len(row) != 1:
        raise ValueError(f'Expected one QC bound for {contrast} {metric}; found {len(row)}')
    return float(row.iloc[0]['absolute_upper_fence'])

metric_specs = [
    ('delta_tsnr', '|Δ tSNR|', 'Absolute tSNR difference'),
    ('delta_fd_mean', '|Δ mean FD| (mm)', 'Absolute mean-FD difference'),
    ('delta_fd_perc', '|Δ volumes with FD > 0.20 mm| (%)', 'Absolute high-motion-percentage difference'),
]
fig, axes = plt.subplots(3, 1, figsize=(14, 15), sharex=True)
for axis, (metric, ylabel, title) in zip(axes, metric_specs):
    plotted = []
    for position, contrast in enumerate(QC_CONTRAST_ORDER, start=1):
        subset = complete_qc.loc[complete_qc['contrast'] == contrast].sort_values('participant').copy()
        values = subset[metric].abs().to_numpy()
        plotted.append(values)
        fence = absolute_fence(contrast, metric)
        mean_fd_flag = subset['delta_fd_mean'].abs().to_numpy() > absolute_fence(contrast, 'delta_fd_mean')
        fd_perc_flag = subset['delta_fd_perc'].abs().to_numpy() > absolute_fence(contrast, 'delta_fd_perc')
        paired_motion = mean_fd_flag & fd_perc_flag
        metric_flag = values > fence
        colors = np.full(len(values), '#6B7280', dtype=object)
        colors[metric_flag] = '#D97706'
        if metric != 'delta_tsnr':
            colors[paired_motion] = '#C73E1D'
        jitter = np.linspace(-0.18, 0.18, len(values))
        axis.scatter(position + jitter, values, c=colors, s=24, alpha=0.86, linewidth=0, zorder=3)
        axis.scatter(position, fence, marker='_', s=260, color='#B91C1C', linewidth=2.0, zorder=4)
        if metric == 'delta_fd_perc':
            for x, value, participant, flagged in zip(position + jitter, values, subset['participant'], paired_motion):
                if flagged:
                    axis.annotate(participant.replace('sub-', ''), (x, value), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=8, color='#8B1E0A')
    boxes = axis.boxplot(plotted, positions=range(1, len(plotted) + 1), widths=0.58, whis=1.5, showfliers=False, patch_artist=True, medianprops={'color': '#111827', 'linewidth': 1.5})
    for box in boxes['boxes']:
        box.set(facecolor='#DDE3EA', edgecolor='#4B5563', alpha=0.66)
    axis.set_ylabel(ylabel)
    axis.set_title(title, fontsize=12, pad=8)
    axis.grid(axis='y', color='#D1D5DB', linewidth=0.7, alpha=0.65)
    axis.spines[['top', 'right']].set_visible(False)
axes[-1].set_xticks(range(1, len(QC_CONTRAST_LABELS) + 1), QC_CONTRAST_LABELS, rotation=24, ha='right')
legend = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#6B7280', markeredgecolor='none', label='Within fence'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#D97706', markeredgecolor='none', label='Metric-specific outlier'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#C73E1D', markeredgecolor='none', label='Paired motion outlier'),
    Line2D([0], [0], marker='_', color='#B91C1C', markersize=15, linewidth=0, label='Tukey upper fence'),
]
fig.suptitle('N=27 condition-level data-quality differences', fontsize=15, y=0.995)
fig.legend(handles=legend, loc='upper center', bbox_to_anchor=(0.5, 0.975), ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
display(fig)
plt.close(fig)

count_columns = [
    'absolute_tsnr_outlier_count',
    'absolute_fd_mean_outlier_count',
    'absolute_fd_perc_outlier_count',
    'paired_motion_outlier_count',
]
for column in count_columns:
    QC_SELECTION[column] = pd.to_numeric(QC_SELECTION[column], errors='raise')
flagged_qc = QC_SELECTION.loc[QC_SELECTION[count_columns].sum(axis=1) > 0, [
    'participant', *count_columns, 'paired_motion_outlier_contrasts', 'decision'
]].copy()
flagged_qc.columns = [
    'Participant', 'tSNR flags', 'Mean-FD flags', 'High-motion-% flags',
    'Paired motion flags', 'Paired flagged comparisons', 'Current sensitivity decision'
]
display(flagged_qc.style.hide(axis='index'))

excluded = QC_SELECTION.loc[QC_SELECTION['decision'] == 'exclude_sensitivity', 'participant'].tolist()
dmn_motion = paired_motion_ids('rtpj-minus-vlpfc')
ecn_motion = paired_motion_ids('both-minus-rtpj')
display(Markdown(
    '### Reading these diagnostics\n\n'
    f'- The documented post hoc sensitivity rule identifies **{', '.join(excluded)}** because each has paired motion flags in at least two planned comparisons.\n'
    '- Single-metric flags are visible but do not trigger that rule. For example, sub-218 has several mean-FD and tSNR difference flags but no paired high-motion-percentage flag.\n'
    f"- The primary DMN finding tests VLPFC versus RTPJ; paired motion flags for that comparison: **{', '.join(dmn_motion) or 'none'}**.\n"
    f"- The primary ECN finding tests RTPJ versus BOTH; paired motion flags for that comparison: **{', '.join(ecn_motion) or 'none'}**.\n"
    '- The seven comparisons share the same four runs, so repeated flags are correlated evidence of a problematic condition/run, not independent QC events.\n'
    '- The seven contrasts were planned, but this exact exclusion rule was defined after viewing these QC distributions. It should therefore be treated as exploratory until the N=28 reconstruction and technical-exclusion history are resolved.'
))

## 6. Interpretation notes

The bars and participant points show values extracted from the stage-2 beta maps. Because each displayed region was selected from the same group contrast, these values are a descriptive visualization of the result, not an independent ROI test. The cluster-extent corrected p-values control spatial inference within each randomise job but do not correct across the 175 primary and secondary network-by-condition jobs; none survives a simple Bonferroni correction across that full family. Confirmatory ROI inference requires an independently defined mask or held-out data.